# Team Challenge SQL
## Parte 1: SQL Murder Mystery - FINAL

### Introducción

En este notebook resolvemos el juego SQL Murder Mystery aplicando un paso a paso limpio.
Se usa la base de datos `data/sql-murder-mystery.db` y se identifican las pruebas, testigos y el asesino.

## Paso 1: Conectar a la base de datos y mostrar tablas

In [13]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('./data/sql-murder-mystery.db')
cursor = conn.cursor()

tablas = cursor.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
print("Tablas disponibles:")
for t in tablas:
    print(f"- {t[0]}")

Tablas disponibles:
- crime_scene_report
- drivers_license
- person
- facebook_event_checkin
- interview
- get_fit_now_member
- get_fit_now_check_in
- income
- solution


## Paso 2: Encontrar el reporte del crimen

In [2]:
query_1 = """
SELECT *
FROM crime_scene_report
WHERE type = 'murder'
  AND date = 20180115
  AND city = 'SQL City'
"""

df_crimen = pd.read_sql_query(query_1, conn)
print(df_crimen.to_string(index=False))

    date   type                                                                                                                                                                               description     city
20180115 murder Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave". SQL City


## Paso 3: Identificar los testigos
1. Vive en Northwestern Dr
2. Annabel (Franklin Ave)

In [3]:
query_2 = """
SELECT p.id, p.name, p.address_number, p.address_street_name, i.transcript
FROM person p
JOIN interview i ON p.id = i.person_id
WHERE p.address_street_name LIKE '%Northwestern Dr%'
ORDER BY p.address_number DESC
LIMIT 1
"""

df_testigo_1 = pd.read_sql_query(query_2, conn)
print("Testigo 1")
print(df_testigo_1.to_string(index=False))

query_3 = """
SELECT p.id, p.name, p.address_number, p.address_street_name, i.transcript
FROM person p
JOIN interview i ON p.id = i.person_id
WHERE p.name LIKE '%Annabel%'
  AND p.address_street_name = 'Franklin Ave'
"""

df_testigo_2 = pd.read_sql_query(query_3, conn)
print("\nTestigo 2")
print(df_testigo_2.to_string(index=False))

Testigo 1
   id           name  address_number address_street_name                                                                                                                                                                                                                      transcript
14887 Morty Schapiro            4919     Northwestern Dr I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".

Testigo 2
   id           name  address_number address_street_name                                                                                                            transcript
16371 Annabel Miller             103        Franklin Ave I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.


## Paso 4: Analizar pistas de los testigos

In [4]:
print("Pista Testigo 1:\n", df_testigo_1['transcript'].values[0])
print("\nPista Testigo 2:\n", df_testigo_2['transcript'].values[0])

# Conclusión preliminar
print("\nPistas recogidas:")
print("- Criminal hombre")
print("- Miembro del gimnasio Get Fit Now")
print("- Clase GOLD con ID '48Z*'")
print("- Estuvo el 09/01/2018")
print("- Placa con 'H42W'")

Pista Testigo 1:
 I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".

Pista Testigo 2:
 I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.

Pistas recogidas:
- Criminal hombre
- Miembro del gimnasio Get Fit Now
- Clase GOLD con ID '48Z*'
- Estuvo el 09/01/2018
- Placa con 'H42W'


## Paso 5: Personas en el gimnasio el 15/01/2018

In [5]:
query_4 = """
SELECT p.id, p.name, d.gender, d.hair_color, d.height, d.plate_number,
       gfm.membership_status, gfc.check_in_time, gfc.check_out_time
FROM person p
JOIN drivers_license d ON p.license_id = d.id
JOIN get_fit_now_member gfm ON p.id = gfm.person_id
JOIN get_fit_now_check_in gfc ON gfm.id = gfc.membership_id
WHERE gfc.check_in_date = 20180115
ORDER BY gfc.check_out_time DESC
"""

df_gym_day = pd.read_sql_query(query_4, conn)
print(df_gym_day.to_string(index=False))

   id             name gender hair_color  height plate_number membership_status  check_in_time  check_out_time
13466     Armando Huie   male      white      65       4R5M5Q           regular           1087            1195
99602 Joline Hollering female     blonde      64       U34X7P              gold            746             836
19948     Taylor Skyes   male      green      67       A33U3B              gold            354             825
50106      Edgar Bamba   male      white      80       S7Z182            silver            525             800


## Paso 6: Filtrar miembros GOLD/PLATINUM ese día

In [6]:
query_5 = """
SELECT p.id, p.name, d.gender, d.hair_color, d.height, d.plate_number,
       gfm.membership_status, gfc.check_in_time, gfc.check_out_time
FROM person p
JOIN drivers_license d ON p.license_id = d.id
JOIN get_fit_now_member gfm ON p.id = gfm.person_id
JOIN get_fit_now_check_in gfc ON gfm.id = gfc.membership_id
WHERE gfc.check_in_date = 20180115
  AND gfm.membership_status IN ('gold','platinum')
ORDER BY gfc.check_out_time DESC
"""

df_premium = pd.read_sql_query(query_5, conn)
print(df_premium.to_string(index=False))

   id             name gender hair_color  height plate_number membership_status  check_in_time  check_out_time
99602 Joline Hollering female     blonde      64       U34X7P              gold            746             836
19948     Taylor Skyes   male      green      67       A33U3B              gold            354             825


## Paso 7: Revisar entrevistas de sospechosos premium

In [7]:
query_6 = """
SELECT p.id, p.name, i.transcript
FROM person p
LEFT JOIN interview i ON p.id = i.person_id
WHERE p.id IN (
    SELECT DISTINCT gfm.person_id
    FROM get_fit_now_member gfm
    JOIN get_fit_now_check_in gfc ON gfm.id = gfc.membership_id
    WHERE gfc.check_in_date = 20180115
      AND gfm.membership_status IN ('gold','platinum')
)
ORDER BY p.name
"""

df_interviews = pd.read_sql_query(query_6, conn)
print(df_interviews.to_string(index=False))

   id             name                          transcript
99602 Joline Hollering                                None
19948     Taylor Skyes   *    *    *    *    *    *    *\n


## Paso 8: Buscar membresías '48Z%' el 09/01/2018

In [8]:
query_8 = """
SELECT *
FROM get_fit_now_check_in
WHERE membership_id LIKE '48Z%'
  AND check_in_date = 20180109
"""

df_gym_suspects = pd.read_sql_query(query_8, conn)
print(df_gym_suspects.to_string(index=False))

membership_id  check_in_date  check_in_time  check_out_time
        48Z7A       20180109           1600            1730
        48Z55       20180109           1530            1700


## Paso 9: Cruce con placa 'H42W' y género masculino

In [9]:
query_9 = """
SELECT *
FROM drivers_license
WHERE gender='male'
  AND plate_number LIKE '%H42W%'
"""

df_drivers = pd.read_sql_query(query_9, conn)
print(df_drivers.to_string(index=False))

    id  age  height eye_color hair_color gender plate_number  car_make car_model
423327   30      70     brown      brown   male       0H42W2 Chevrolet  Spark LS
664760   21      71     black      black   male       4H42WR    Nissan    Altima


## Paso 10: Personas con esas licencias

In [10]:
if len(df_drivers) > 0:
    license_ids = ', '.join([f"{int(x)}" for x in df_drivers['id']])
    query_10 = f"""
SELECT *
FROM person
WHERE license_id IN ({license_ids})
"""
    df_persons = pd.read_sql_query(query_10, conn)
    print(df_persons.to_string(index=False))
else:
    print('No hay conductores que cumplan criterio.')

   id           name  license_id  address_number   address_street_name       ssn
51739 Tushar Chandra      664760             312                Phi St 137882671
67318  Jeremy Bowers      423327             530 Washington Pl, Apt 3A 871539279


## Paso 11: Identificar sospechoso en membros del gym

In [11]:
if len(df_persons) > 0:
    person_ids = ', '.join([f"{int(x)}" for x in df_persons['id']])
    query_11 = f"""
SELECT *
FROM get_fit_now_member
WHERE person_id IN ({person_ids})
"""
    df_murderer = pd.read_sql_query(query_11, conn)
    print(df_murderer.to_string(index=False))
else:
    df_murderer = pd.DataFrame()
    print('No se encontraron personas sospechosas en person table.')

   id  person_id          name  membership_start_date membership_status
48Z55      67318 Jeremy Bowers               20160101              gold


## Paso 12: Confirmación en tabla `solution`

In [12]:
if len(df_murderer) == 1:
    murderer_name = df_murderer['name'].iloc[0]
    conn.execute('INSERT INTO solution VALUES (1, ?)', (murderer_name,))
    conn.commit()
    df_solution = pd.read_sql_query('SELECT * FROM solution', conn)
    print('Solución confirmada:', df_solution.to_string(index=False))
else:
    print('No se insertó solución porque hay más de un sospechoso o ninguno.')

conn.close()

Solución confirmada:  user                                                                                                                                                                                                                                                                                                                                                                                              value
    0 Congrats, you found the murderer! But wait, there's more... If you think you're up for a challenge, try querying the interview transcript of the murderer to find the real villain behind this crime. If you feel especially confident in your SQL skills, try to complete this final step with no more than 2 queries. Use this same INSERT statement with your new suspect to check your answer.


## Conclusión

El asesino identificado es **Jeremy Bowers** 